Example Spak SQL

Name: Emilio Seda

In [1]:
from spark_utils import SparkUtils
spark_url = "spark://spark-master:7077"
app_name = "Spark SQL - Example 1"
su = SparkUtils(spark_url, app_name)
su

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/19 00:20:09 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


<SparkContext master=local[*] appName=Spark SQL example>

In [2]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

data = [
    (1, "Alice", 29),
    (2, "Bob", 35),
    (3, "Charlie", 41)]

schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("age", IntegerType(), True)
])


df = su._spark.createDataFrame(data, schema)
df.printSchema()

root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- age: integer (nullable = true)



In [3]:

from pyspark.sql.types import StructType, StructField, StringType, TimestampType, FloatType
from datetime import datetime

from pyspark.sql.functions import col

factory_data = [
("M001", datetime(2026, 1, 26, 8, 0, 0), 75.3),
("M002", datetime(2026, 1, 26, 8, 5, 0), 68.7),
("M001", datetime(2026, 1, 26, 8, 10, 0), 76.1),
("M003", datetime(2026, 1, 26, 8, 15, 0), 72.4),
("M002", datetime(2026, 1, 26, 8, 20, 0), 69.8),
("M001", datetime(2026, 1, 26, 8, 25, 0), 77.5),
("M003", datetime(2026, 1, 26, 8, 30, 0), 73.2),
("M002", datetime(2026, 1, 26, 8, 35, 0), 70.1),
("M001", datetime(2026, 1, 26, 8, 40, 0), 78.0),
("M003", datetime(2026, 1, 26, 8, 45, 0), 74.6),
]

schema = StructType([
    StructField("id",StringType(), True),
    StructField("timestamp",TimestampType(), True),
    StructField("temperature",FloatType(), True)
])

factory_df = su._spark.createDataFrame(factory_data, schema)
factory_df.printSchema()


record_count = factory_df.count()
print(f"number of records: {record_count}")

filtered_df = factory_df.filter(col("temperature") > 40)
filtered_df.show(truncate=False)


root
 |-- id: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- temperature: float (nullable = true)



number of records: 10
+----+-------------------+-----------+
|id  |timestamp          |temperature|
+----+-------------------+-----------+
|M001|2026-01-26 08:00:00|75.3       |
|M002|2026-01-26 08:05:00|68.7       |
|M001|2026-01-26 08:10:00|76.1       |
|M003|2026-01-26 08:15:00|72.4       |
|M002|2026-01-26 08:20:00|69.8       |
|M001|2026-01-26 08:25:00|77.5       |
|M003|2026-01-26 08:30:00|73.2       |
|M002|2026-01-26 08:35:00|70.1       |
|M001|2026-01-26 08:40:00|78.0       |
|M003|2026-01-26 08:45:00|74.6       |
+----+-------------------+-----------+



In [5]:
#Get the average temperature per machine
from pyspark.sql.functions import avg

agg_df = factory_df.groupBy("id").agg(
    avg("temperature").alias("avg_temp")
)

agg_df.show()




[Stage 6:==============>                                          (4 + 12) / 16]

+----+-----------------+
|  id|         avg_temp|
+----+-----------------+
|M001|76.72500038146973|
|M002|69.53333282470703|
|M003|73.39999898274739|
+----+-----------------+



In [6]:
#Find the maximum and minimum temperature per machine
from pyspark.sql.functions import min, max

agg_df = factory_df.groupBy("id").agg(
    min("temperature").alias("min_temp"),
    max("temperature").alias("max_temp")
)

agg_df.show()


[Stage 9:>                                                        (0 + 16) / 16]

+----+--------+--------+
|  id|min_temp|max_temp|
+----+--------+--------+
|M001|    75.3|    78.0|
|M002|    68.7|    70.1|
|M003|    72.4|    74.6|
+----+--------+--------+



In [10]:
#Filter records above a temperature threshold (temp > 75).

filtered_df = factory_df.filter(col("temperature") > 75)
filtered_df.show()



+----+-------------------+-----------+
|  id|          timestamp|temperature|
+----+-------------------+-----------+
|M001|2026-01-26 08:00:00|       75.3|
|M001|2026-01-26 08:10:00|       76.1|
|M001|2026-01-26 08:25:00|       77.5|
|M001|2026-01-26 08:40:00|       78.0|
+----+-------------------+-----------+



In [12]:
#count of reading per machine
grouped_df = factory_df.groupBy("id").count()
grouped_df.show()


[Stage 18:=============================================>          (13 + 3) / 16]

+----+-----+
|  id|count|
+----+-----+
|M001|    4|
|M002|    3|
|M003|    3|
+----+-----+



In [14]:
#Machine with the highest temperature

agg_df = factory_df.orderBy(col("temperature").desc()).limit(1)

agg_df.show()


[Stage 24:====================================================>   (15 + 1) / 16]

+----+-------------------+-----------+
|  id|          timestamp|temperature|
+----+-------------------+-----------+
|M001|2026-01-26 08:40:00|       78.0|
+----+-------------------+-----------+



In [16]:
su._spark.stop()